In [1]:
#| hide
%load_ext autoreload
%autoreload 2

# InstructorParser

> Using OpenAI and instructor to parse the affiliation information

In [2]:
#| default_exp instructor_parser

In [3]:
#| hide
from nbdev.showdoc import *
from dotenv import load_dotenv

In [4]:
#| hide
load_dotenv()

True

In [5]:
#| export
import os

In [6]:
#| export
from pydantic import BaseModel, model_validator, Field, field_validator, EmailStr
from typing import Any
import openai

In [7]:
class Author(BaseModel):
    last_name:str = Field(description="The last name of the author")
    fore_name:str = Field(description="The first name of the author")
    initials:str = Field(description="The initials of the author")
    email:EmailStr = Field(description="The email of the author")
    identifier:str = Field(description="The identifier of the author")

In [8]:
#| export
class Affiliation(BaseModel):
    """Affiliation data representation"""
    author:Author = Field(description="The author of the affiliation")
    organization:str = Field(
        default= "",
        description= "The company or university of the Affiliation",        
    )
    laboratory: str = Field(
        default= "",
        description= "The Laboratory of the Affiliation",        
    )
    department: str =Field(
        default= "",
        description= "The department of the Affiliation",        
    )
    faculty: str =Field(
        default= "",
        description= "The faculty of the Affiliation",        
    )
    country: str = Field(
        default= "",
        description= "The country of the Affiliation",        
    )
    city: str = Field(
        default= "",
        description= "The city of the Affiliation",        
    )
    state: str = Field(
        default= "",
        description= "The state, county, or province of the Affiliation",        
    )
    email: EmailStr | None= Field(
        default= None,
        description= "The email Address",        
    )
    

In [12]:
#| export
from pydantic import ValidationInfo


class InstructorParser(BaseModel):
    """Class to activate the OpenAI Parser algorithm"""
    data_class: Any
    model: str = 'qwen3:14b'
    client:Any

    @model_validator(mode='before')
    @classmethod
    def validate_env(cls, values:dict, info:ValidationInfo):
        try:
            import instructor
            import openai
        except ImportError:
            raise ImportError(
                "Could not import instructor python package. "
                "This is needed in order to accurately extract the data "
                "from Affiliations. Please install it with `pip install instructor`."
            )

        if not values['data_class']:
            raise Exception("It is needed a data class to guide the LLM to extract information")
        client = openai.OpenAI(api_key="ollama",base_url="http://localhost:11434/v1")
        values['client'] = instructor.patch(client, mode=instructor.Mode.JSON)
        return values
    
    def run(self, text):
        data = self.client.chat.completions.create(
            model=self.model,
            response_model=self.data_class,
            messages=[{
                "role": "system", 
                "content": "You are getting information from pubmed articles. You are given a list of authors and their affiliations. You need to extract the information and return it in a structured format."
            }, {
                "role": "user", 
                "content": text
            }])
        return data

    def to_dict(self):
        return self.data_class.model_dump()

In [13]:
i_client = InstructorParser(data_class=Affiliation)

In [14]:
i_client.run("{'Identifier': [], 'AffiliationInfo': [{'Identifier': [], 'Affiliation': 'Department of Medical Biochemistry and Biophysics, Karolinska Institutet, S-171 77 Stockholm, Sweden.'}], 'LastName': 'Stenmark', 'ForeName': 'Pål', 'Initials': 'P'}, attributes={'ValidYN': 'Y'}")

ValidationError: 1 validation error for Affiliation
author
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.6/v/missing

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()